In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/carlodemello/working-version-of-code/Complete working model/image_classifier_model (1).onnx
/kaggle/input/datasets/carlodemello/working-version-of-code/Complete working model/CNN_best (1).pth
/kaggle/input/datasets/carlodemello/working-version-of-code/Complete working model/best_model (4).pth
/kaggle/input/datasets/carlodemello/working-version-of-code/Complete working model/activations (4).pt
/kaggle/input/datasets/carlodemello/working-version-of-code/Complete working model/train_dataset.pt
/kaggle/input/datasets/carlodemello/working-version-of-code/Complete working model/checkpoints_zip (6)/my_model_epoch_5.pth
/kaggle/input/datasets/carlodemello/working-version-of-code/Complete working model/checkpoints_zip (6)/my_model_epoch_6.pth
/kaggle/input/datasets/carlodemello/working-version-of-code/Complete working model/checkpoints_zip (6)/CNN_epoch_5.pth
/kaggle/input/datasets/carlodemello/working-version-of-code/Complete working model/checkpoints_zip (6)/my_model_ep

In [2]:
import torch 



In [3]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import matplotlib.pyplot as plt 
import pandas as pd
import numpy as np
import os 
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import onnx

In [4]:
model1 = torch.load("/kaggle/input/datasets/carlodemello/working-version-of-code/Complete working model/checkpoints_zip (6)/my_model_epoch_1.pth")

In [5]:
class ShapesCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 3)
        )

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)

    def forward(self, x):
        activations = {}   # dictionary: layer_name → output

        # conv layers
        for i, layer in enumerate(self.conv_layers):
            x = layer(x)
            activations[f"conv_{i}_{layer.__class__.__name__}"] = x.detach()

        # classifier layers
        for i, layer in enumerate(self.classifier):
            x = layer(x)
            activations[f"classifier_{i}_{layer.__class__.__name__}"] = x.detach()

        return x, activations

The following code, is to put the checkpoint data in for the model. i.e. loadign the checkpoint data into the model. 

In [6]:
import torch
import torch.nn as nn

model = ShapesCNN()
checkpoint = torch.load("/kaggle/input/datasets/carlodemello/working-version-of-code/Complete working model/checkpoints_zip (6)/my_model_epoch_1.pth")
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print("Model loaded successfully!")

Model loaded successfully!


In [7]:
model

ShapesCNN(
  (conv_layers): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=2048, out_features=256, bias=True)
    (2): ReLU()
    (3): Dro

In [8]:
0,1,4,5,8,9

# Classifier 

1,4,7,# Can be visualisewd.

(1, 4, 7)

In [9]:
model.conv_layers[0].weight.mean()


tensor(-0.0042, grad_fn=<MeanBackward0>)

In [10]:
weight_layer0 = model.conv_layers[0].weight.detach().cpu()
weight_layer1 = model.conv_layers[1].weight.detach().cpu()
weight_layer2 = model.conv_layers[4].weight.detach().cpu()
weight_layer3 = model.conv_layers[5].weight.detach().cpu()
weight_layer4 = model.conv_layers[8].weight.detach().cpu()
weight_layer5 = model.conv_layers[9].weight.detach().cpu()

In [11]:
weight_0_mean = weight_layer0.numpy().mean()
weight_1_mean = weight_layer1.numpy().mean()
weight_2_mean = weight_layer2.numpy().mean()
weight_3_mean = weight_layer3.numpy().mean()
weight_4_mean = weight_layer4.numpy().mean()
weight_5_mean = weight_layer5.numpy().mean()

In [12]:
# Specify the conv layer indices you want to extract
layer_indices = [0, 1, 4, 5, 8, 9]

# Build the dictionary automatically
weights_dict = {
    f"conv{idx}": model.conv_layers[idx].weight.detach().cpu()
    for idx in layer_indices
}


In [13]:
weight1_means = {layer: weights_dict[layer].numpy().mean() for layer in weights_dict}

In [14]:
weight1_std = {layer: weights_dict[layer].numpy().std() for layer in weights_dict}

In [15]:
weight_std

NameError: name 'weight_std' is not defined

In [ ]:
model2 = ShapesCNN()
checkpoint = torch.load("/kaggle/input/datasets/carlodemello/working-version-of-code/Complete working model/checkpoints_zip (6)/my_model_epoch_9.pth")
model2.load_state_dict(checkpoint['model_state_dict'])
model2.eval()
print("Model 2 loaded successfully!")

In [ ]:
# Specify the conv layer indices you want to extract
layer_indices = [0, 1, 4, 5, 8, 9]

# Build the dictionary automatically
weights2_dict = {
    f"conv{idx}": model2.conv_layers[idx].weight.detach().cpu()
    for idx in layer_indices
}

In [ ]:
weight2_means = {layer: weights_dict[layer].numpy().mean() for layer in weights_dict}
weight2_std = {layer: weights_dict[layer].numpy().std() for layer in weights_dict}

In [ ]:
weight_means

In [ ]:
bias_layer0 = model.conv_layers[0].bias

In [ ]:
bias_layer0 = model.conv_layers[0].bias.detach().cpu()
bias_layer1 = model.conv_layers[1].bias.detach().cpu()
bias_layer2 = model.conv_layers[4].bias.detach().cpu()
bias_layer3 = model.conv_layers[5].bias.detach().cpu()
bias_layer4 = model.conv_layers[8].bias.detach().cpu()
bias_layer5 = model.conv_layers[9].bias.detach().cpu()

In [ ]:
layer_indices = [0, 1, 4, 5, 8, 9]

# Build the dictionary automatically
bias_dict = {
    f"conv{idx}": model.conv_layers[idx].bias.detach().cpu()
    for idx in layer_indices
}


In [ ]:
bias_dict

In [ ]:
bias_means = {layer: bias_dict[layer].numpy().mean() for layer in bias_dict}

In [ ]:
bias_means

In [ ]:
class_indx = [1,4,7]

classifier_weight_dict = {
    f"conv{idx}": model.classifier[idx].weight.detach().cpu()
    for idx in class_indx
}

classifier_bias_dict = {
    f"conv{idx}": model.classifier[idx].bias.detach().cpu()
    for idx in class_indx
}

In [ ]:
classifier_weight_dict

In [ ]:
classifier_bias_dict

In [ ]:
# Layer 2 and 10 are the relu data 

conv